In [ ]:
## Devin SaJeun DuCharme
## CS790 Homework 3
## JAX JIT Deep Dive

In [ ]:
# Imports
import time
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

In [ ]:
## Part 1: Measuring Compilation Overhead

# Elementwise operations on large matrix
def elementmatrix(x):
    x = x + 1
    x = x * 2
    x = jnp.sin(x)
    x = jnp.exp(x)
    x = jnp.tanh(x)
    x = jnp.square(x)
    x = jnp.sqrt(x + 1)
    x = jnp.log(x + 1)
    x = x - 3
    x = jnp.abs(x)
    return x

# JIT config
jitelementmatrix = jax.jit(elementmatrix)

# Timer
def measure_time(function_to_run, input_matrix):
    start_time = time.perf_counter()
    
    result = function_to_run(input_matrix)
    result.block_until_ready()
    
    end_time = time.perf_counter()
    
    return end_time - start_time

# Matrix sizes and version creation
matrix_sizes = [100, 500, 1000, 5000]

eagerm = []
firstm = []
secondm = []

# Three versions execution
for size in matrix_sizes:
    print(f"\nNow testing matrix size: {size} x {size}")
    
    input_matrix = jnp.ones((size, size), dtype=jnp.float32)
    
    # Eager mode
    eager = measure_time(elementmatrix, input_matrix)
    eagerm.append(eager)
    
    # First JIT call
    first = measure_time(jitelementmatrix, input_matrix)
    firstm.append(first)
    
    # Second JIT call
    second = measure_time(jitelementmatrix, input_matrix)
    secondm.append(second)
    
    print(f"Eager mode time: {eager:.6f} seconds")
    print(f"First JIT call time: {first:.6f} seconds")
    print(f"Second JIT call time: {second:.6f} seconds")

# Summary plots for each  

plt.figure(figsize=(8, 5))

plt.plot(matrix_sizes, eagerm, marker='o', label='Eager mode')
plt.plot(matrix_sizes, firstm, marker='o', label='First JIT call')
plt.plot(matrix_sizes, secondm, marker='o', label='Second JIT call')

plt.xlabel('Matrix Size (N x N)')
plt.ylabel('Execution Time (seconds)')
plt.title('JAX Execution Time vs Matrix Size')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
## Part 2: Shape Specialization

# Row mean function
def row_mean(matrix):
    return jnp.mean(matrix, axis=1)

# JIT config
jit_row_mean = jax.jit(row_mean)

# Timer
def measure_time(function_to_run, input_matrix):
    start_time = time.perf_counter()
    
    result = function_to_run(input_matrix)
    result.block_until_ready()
    
    end_time = time.perf_counter()
    
    return end_time - start_time

# Matrix shapes
matrix_a = jnp.ones((100, 100), dtype=jnp.float32)
matrix_b = jnp.ones((100, 200), dtype=jnp.float32)
matrix_c = jnp.ones((100, 100), dtype=jnp.float32)
matrix_d = jnp.ones((200, 100), dtype=jnp.float32)

# Labels for plotting
labels = [
    "(100,100) Initial",
    "(100,200) Altered",
    "(100,100) Repeat Initial",
    "(200,100) Altered V2"
]

times = []

# Execution with different shapes

print("\nTesting (100,100) first call")
t1 = measure_time(jit_row_mean, matrix_a)
times.append(t1)
print(f"Time: {t1:.6f} seconds")

print("\nTesting (100,200) new shape")
t2 = measure_time(jit_row_mean, matrix_b)
times.append(t2)
print(f"Time: {t2:.6f} seconds")

print("\nTesting (100,100) repeated shape")
t3 = measure_time(jit_row_mean, matrix_c)
times.append(t3)
print(f"Time: {t3:.6f} seconds")

print("\nTesting (200,100) new shape")
t4 = measure_time(jit_row_mean, matrix_d)
times.append(t4)
print(f"Time: {t4:.6f} seconds")

# Inspect JAX tracing behavior

print("\nJAXPR for (100,100):")
print(jax.make_jaxpr(row_mean)(matrix_a))

print("\nJAXPR for (100,200):")
print(jax.make_jaxpr(row_mean)(matrix_b))

print("\nJAXPR for (200,100):")
print(jax.make_jaxpr(row_mean)(matrix_d))

# Bar chart visualization

plt.figure(figsize=(8, 5))
plt.bar(labels, times)

plt.ylabel("Execution Time (seconds)")
plt.title("JAX JIT Timing for Different Input Shapes")
plt.xticks(rotation=20)
plt.grid(axis='y')

plt.show()

In [ ]:
## Part 3: Operator Fusion Analysis

# Imports
import time
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Input array matrix
x = jnp.ones((1000, 1000), dtype=jnp.float32)

# Timer
def measure_time(function_to_run, input_value):
    start_time = time.perf_counter()
    
    result = function_to_run(input_value)
    result.block_until_ready()
    
    end_time = time.perf_counter()
    
    return end_time - start_time

# Apply sin i times
def repeated_sin(x, i):
    result = x
    for _ in range(i):
        result = jnp.sin(result)
    return result

# Apply cos i times
def repeated_cos(x, i):
    result = x
    for _ in range(i):
        result = jnp.cos(result)
    return result

# Version A: separate operations in Python loop (eager)
def version_a(x):
    total = jnp.zeros_like(x)
    
    for i in range(1, 101):
        sin_part = repeated_sin(x, i)
        cos_part = repeated_cos(x, i)
        total = total + sin_part + cos_part
    
    return total

# Version B: single composed expression for JIT compilation
def version_b(x):
    terms = [
        repeated_sin(x, i) + repeated_cos(x, i)
        for i in range(1, 101)
    ]
    
    return jnp.sum(jnp.stack(terms), axis=0)

# JIT config
jit_version_b = jax.jit(version_b)

# Timing results
eager_time = measure_time(version_a, x)
first_jit_time = measure_time(jit_version_b, x)
second_jit_time = measure_time(jit_version_b, x)

print(f"Eager version time: {eager_time:.6f} seconds")
print(f"First JIT version time: {first_jit_time:.6f} seconds")
print(f"Second JIT version time: {second_jit_time:.6f} seconds")

# Plot comparison
labels = ['Eager version', 'First JIT call', 'Second JIT call']
times = [eager_time, first_jit_time, second_jit_time]

plt.figure(figsize=(8, 5))
plt.bar(labels, times)

plt.ylabel("Execution Time (seconds)")
plt.title("Operator Fusion Timing Comparison")
plt.grid(axis='y')

plt.show()